# Step1: Preparing Dataset

<div class="alert alert-block alert-success">

In this section, we download and format the instruction dataset for instruction finetuning a
pretrained LLM in this chapter. The dataset consists of 1100 instruction-response pairs.

The following code implements and executes a function to download this dataset, which
is a relatively small file, only 204 KB in size, in JSON format. JSON, or JavaScript Object
Notation, mirrors the structure of Python dictionaries, providing a simple structure for data
interchange that is both human-readable and machine-friendly.

</div>

In [20]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

In [21]:
file_path = "../data/instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


<div class="alert alert-block alert-success">

The data list , which we loaded from the JSON file contains the 1100 entries of the
instruction dataset. 

Let's print one of the entries to see how each entry is structured:

</div>

In [22]:
print(f"Example entry: {data[50]}")

Example entry: {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [23]:
print(f"Example entry: {data[900]}")

Example entry: {'instruction': 'Rewrite the sentence using an idiom.', 'input': 'He is very rich.', 'output': 'He is rolling in dough.'}


## Converting Instructions into Alpaca Format

In [24]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task, paired with an input that provides further context."
        f"Write a response that appropriately completes the request."
        f"### Instruction:\n {entry['instruction']}:"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

<div class="alert alert-block alert-info">
    
This format_input function takes a dictionary entry as input and constructs a formatted
string.

</div>

<div class="alert alert-block alert-success">

 Let's test it to dataset entry data[50], which to looked at earlier:

</div>

In [25]:
print(format_input(data[50]))

Below is an instruction that describes a task, paired with an input that provides further context.Write a response that appropriately completes the request.### Instruction:
 Identify the correct spelling of the following word.:

### Input:
Ocassion


In [26]:
model_input = format_input(data[50])
desired_response = f"\n\n Response: \n {data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task, paired with an input that provides further context.Write a response that appropriately completes the request.### Instruction:
 Identify the correct spelling of the following word.:

### Input:
Ocassion

 Response: 
 The correct spelling is 'Occasion.'


<div class="alert alert-block alert-warning">

Note that the format_input skips the optional ### Input: section if the 'input' field is
empty, which we can test out by applying the format_input function to entry data[999]
that we inspected earlier:

</div>

In [27]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task, paired with an input that provides further context.Write a response that appropriately completes the request.### Instruction:
 What is an antonym of 'complicated'?:

### Response:
An antonym of 'complicated' is 'simple'.


## Splitting Dataset into Train-test-validation

In [28]:
train_portion = int (len(data) * 0.85) # 85% for training
test_portion = int(len(data)*0.1) # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation



train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]


In [29]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


<div class="alert alert-block alert-warning">

Having successfully downloaded and partitioned the dataset, and gained a clear
understanding of the dataset prompt formatting, we are now ready for the core
implementation of the instruction finetuning process.

</div>

## Step2: Organizing Data into training batches

<div class="alert alert-block alert-success">

In the previous chapter, the training batches were created automatically by the PyTorch
DataLoader class, which employs a default collate function to combine lists of samples into
batches. 

A collate function is responsible for taking a list of individual data samples and
merging them into a single batch that can be processed efficiently by the model during
training.

</div>

<div class="alert alert-block alert-warning">

However, the batching process for instruction finetuning in this chapter is a bit more
involved and requires us to create our own custom collate function that we will later plug
into the DataLoader. 

We implement this custom collate function to handle the specific
requirements and formatting of our instruction finetuning dataset.

</div>

In [32]:

import torch
import torch.nn as nn


AttributeError: partially initialized module 'torch' has no attribute 'nn' (most likely due to a circular import)

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer(full_text))

